# Archived research notebook

Original filename: `preprocessing first注释（适配PNXX数据集）.ipynb`  
Purpose: Annotate PNXX recordings from seizure lists.  
Historical research version retained for traceability. The original test data are unavailable; legacy outputs, execution state, and attachments were removed for public release.


In [ ]:
import os
import mne
from collections import defaultdict
from datetime import datetime  # 用于判断类型

# 解析摘要文件获取癫痫发作信息
def parse_seizure_summary(summary_path):
    seizure_info = defaultdict(list)
    current_file = None
    file_start_time = None
    
    with open(summary_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # 提取EDF文件名
            if line.startswith('File name:'):
                current_file = line.split(': ')[1].strip()
                file_start_time = None
            # 记录文件开始时间，用于计算相对秒数
            elif line.startswith('Registration start time:') and current_file:
                start_time_str = line.split(': ')[1].strip()
                file_start_time = datetime.strptime(start_time_str, '%H.%M.%S')
            # 提取发作起始时间
            elif line.startswith('Seizure start time:') and current_file and file_start_time:
                seizure_time_str = line.split(': ')[1].strip()
                seizure_time = datetime.strptime(seizure_time_str, '%H.%M.%S')
                
                # 计算相对秒数
                file_seconds = file_start_time.hour * 3600 + file_start_time.minute * 60 + file_start_time.second
                seizure_seconds = seizure_time.hour * 3600 + seizure_time.minute * 60 + seizure_time.second
                if seizure_seconds < file_seconds:  # 处理跨天情况
                    seizure_seconds += 24 * 3600
                start_sec = float(seizure_seconds - file_seconds)
            # 提取发作结束时间
            elif line.startswith('Seizure end time:') and current_file and file_start_time:
                seizure_time_str = line.split(': ')[1].strip()
                seizure_time = datetime.strptime(seizure_time_str, '%H.%M.%S')
                
                # 计算相对秒数
                file_seconds = file_start_time.hour * 3600 + file_start_time.minute * 60 + file_start_time.second
                seizure_seconds = seizure_time.hour * 3600 + seizure_time.minute * 60 + seizure_time.second
                if seizure_seconds < file_seconds:  # 处理跨天情况
                    seizure_seconds += 24 * 3600
                end_sec = float(seizure_seconds - file_seconds)
                
                # 添加发作信息
                seizure_info[current_file].append((start_sec, end_sec))
    
    return seizure_info

# 为 EDF 文件添加发作注释并保存为 FIF 文件
def annotate_edf_files(folder_path, summary_path):
    seizure_info = parse_seizure_summary(summary_path)

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.edf'):
                edf_path = os.path.join(root, file)
                edf_name = file
                seizures = seizure_info.get(edf_name, [])

                if not seizures:
                    print(f"跳过 {edf_name}：无发作信息")
                    continue

                try:
                    # 读取 EDF 文件（preload=True 确保注释正确关联时间）
                    raw = mne.io.read_raw_edf(edf_path, preload=True)
                    
                    # -------------------- 处理 datetime 类型的 meas_date --------------------
                    meas_date = raw.info.get('meas_date')
                    if isinstance(meas_date, datetime):
                        # 将 datetime 对象转换为 Unix 时间戳（整数秒）
                        meas_date = int(meas_date.timestamp())
                        print(f"检测到 datetime 时间戳，转换为 {meas_date}")
                    elif isinstance(meas_date, (float, int)):
                        meas_date = int(meas_date)  # 确保为整数
                    else:
                        meas_date = 0  # 其他情况设为0

                    # 校验时间戳是否在有效范围内
                    if not (-2147483648 <= meas_date <= 2147483647):
                        print(f"重置无效时间戳 {meas_date} 为 0")
                        meas_date = 0
                    raw.set_meas_date(meas_date)
                    # -------------------- 其他逻辑保持不变 --------------------

                    # 创建注释对象（使用相对时间，基于文件开头的秒数）
                    onsets = [start for start, end in seizures]
                    durations = [end - start for start, end in seizures]
                    descriptions = ['seizure'] * len(seizures)
                    annotations = mne.Annotations(
                        onset=onsets,
                        duration=durations,
                        description=descriptions,
                        orig_time=meas_date  # 使用转换后的整数时间戳
                    )

                    # 添加注释
                    raw.set_annotations(annotations)
                    print(f"成功为 {edf_name} 添加 {len(seizures)} 条发作注释")

                    # 保存为符合 MNE 规范的文件名
                    fif_name = edf_name.replace('.edf', '_raw.fif')
                    fif_path = os.path.join(root, fif_name)
                    raw.save(fif_path, overwrite=True)
                    print(f"已保存为 {os.path.basename(fif_path)}")

                except Exception as e:
                    print(f"处理 {edf_name} 时出错: {e}")

# 主函数
if __name__ == "__main__":
    folder_path = r"data/pnxx/PN00"  # EDF 文件路径
    summary_path = r"data/pnxx/PN00/Seizures-list-PN00.txt"  # 摘要文件路径

    annotate_edf_files(folder_path, summary_path)
    print("所有文件处理完成") 